# 07 — Scikit-learn: ML Basics End-to-End

The full supervised-learning workflow in one notebook: load → explore → preprocess → train → evaluate → tune. We cover a regression and a classification problem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import datasets, metrics
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report, ConfusionMatrixDisplay
)
sns.set_theme(style='whitegrid')
rng = np.random.default_rng(0)
print('scikit-learn OK')

## Part A — Regression: Boston-style synthetic housing data

We create a synthetic dataset so every step is reproducible with no internet needed.

In [ ]:
# Synthetic dataset: 500 houses, 4 features → price
n = 500
size    = rng.normal(1500, 400, n).clip(500, 3500)
rooms   = rng.integers(2, 8, n).astype(float)
age     = rng.uniform(0, 50, n)
dist_cbd = rng.uniform(1, 30, n)
noise   = rng.normal(0, 25_000, n)

price = 120 * size + 15_000 * rooms - 800 * age - 5_000 * dist_cbd + 50_000 + noise

housing = pd.DataFrame({
    'size_sqft' : size,
    'rooms'     : rooms,
    'age_years' : age,
    'dist_cbd'  : dist_cbd,
    'price'     : price,
})
housing.describe().round(0)

### 1. Exploratory peek

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, ['size_sqft','rooms','age_years','dist_cbd']):
    ax.scatter(housing[col], housing['price'], alpha=0.2, s=10)
    ax.set_xlabel(col); ax.set_ylabel('price')
plt.suptitle('Feature vs Price', y=1.02)
plt.tight_layout()
plt.show()

### 2. Train / test split

In [ ]:
X = housing.drop(columns='price')
y = housing['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

### 3. Pipeline: scaler + model
Always scale *inside* a pipeline so test-set statistics never leak into training.

In [ ]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  LinearRegression()),
])
pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
r2   = r2_score(y_test, y_pred)
print(f'Linear Regression → RMSE: ${rmse:,.0f}  |  R²: {r2:.3f}')

In [ ]:
# Compare with Ridge and GradientBoosting
pipe_ridge = Pipeline([('sc', StandardScaler()), ('m', Ridge(alpha=10))])
pipe_gb    = Pipeline([('sc', StandardScaler()),
                        ('m', GradientBoostingRegressor(n_estimators=100, random_state=0))])

for name, pipe in [('Ridge', pipe_ridge), ('GradBoost', pipe_gb)]:
    pipe.fit(X_train, y_train)
    p = pipe.predict(X_test)
    print(f'{name:12s} → RMSE: ${mean_squared_error(y_test, p, squared=False):,.0f}  '
          f'R²: {r2_score(y_test, p):.3f}')

### 4. Residual plot — diagnosing model quality

In [ ]:
residuals = y_test - pipe_gb.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(pipe_gb.predict(X_test), residuals, alpha=0.4, s=15)
axes[0].axhline(0, color='red', lw=1)
axes[0].set_xlabel('Predicted price'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Fitted')

axes[1].hist(residuals, bins=30, edgecolor='white', color='steelblue')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## Part B — Classification: Iris dataset

Switching to a classic multi-class problem.

In [ ]:
iris = datasets.load_iris()
X_ir = pd.DataFrame(iris.data,   columns=iris.feature_names)
y_ir = pd.Series(iris.target_names[iris.target], name='species')

print(X_ir.shape)
print(y_ir.value_counts())

### 5. Train/test split & baseline accuracy

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_ir, y_ir, test_size=0.25,
                                               stratify=y_ir, random_state=0)
print(f'Train: {X_tr.shape} | Test: {X_te.shape}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=4, random_state=0),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=0),
}

results = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, m.predict(X_te))
    cv  = cross_val_score(m, X_ir, y_ir, cv=5, scoring='accuracy').mean()
    results[name] = {'test_acc': acc, '5-fold_cv': cv}
    print(f'{name:22s} test={acc:.3f}  cv={cv:.3f}')

pd.DataFrame(results).T.round(3)

### 6. Confusion matrix & classification report

In [ ]:
best = models['Random Forest']
y_pred_ir = best.predict(X_te)

print(classification_report(y_te, y_pred_ir))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_te, y_pred_ir, ax=ax, colorbar=False,
                                         cmap='Blues')
ax.set_title('Random Forest — Confusion Matrix')
plt.tight_layout()
plt.show()

### 7. Feature importance

In [ ]:
importances = pd.Series(
    best.feature_importances_,
    index=iris.feature_names
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(6, 3))
importances.plot.barh(ax=ax, color='steelblue')
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Mean decrease in impurity')
plt.tight_layout()
plt.show()

### 8. Hyperparameter tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth'   : [None, 3, 5],
    'min_samples_split': [2, 5],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=0),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
)
grid.fit(X_tr, y_tr)

print('Best params:', grid.best_params_)
print('Best CV acc:', grid.best_score_.round(4))
print('Test acc   :', accuracy_score(y_te, grid.predict(X_te)).round(4))

### 9. Preprocessing heterogeneous data (ColumnTransformer)
Real datasets mix numeric and categorical columns.

In [ ]:
# Attach a fake categorical column to Iris for demo
df_ir = X_ir.copy()
df_ir['location'] = rng.choice(['urban', 'rural', 'suburban'], size=len(df_ir))

numeric_cols     = list(iris.feature_names)
categorical_cols = ['location']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(),   numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
])

pipe_full = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=0)),
])

cv_scores = cross_val_score(pipe_full, df_ir, y_ir, cv=5, scoring='accuracy')
print(f'CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## Key scikit-learn API cheat sheet

| Method | Purpose |
|---|---|
| `fit(X, y)` | Train the model |
| `predict(X)` | Hard predictions |
| `predict_proba(X)` | Probability estimates |
| `score(X, y)` | Default metric |
| `cross_val_score(...)` | k-fold CV |
| `GridSearchCV(...)` | Hyperparameter search |
| `Pipeline([...])` | Chain steps safely |

## Exercises
1. Replace the GradientBoosting regressor with a `RandomForestRegressor`. Compare RMSE and R².
2. Add polynomial features (`sklearn.preprocessing.PolynomialFeatures`, degree=2) to the housing pipeline before the scaler and report the change in RMSE.
3. On the Iris data, use `GridSearchCV` on a `LogisticRegression` varying `C` in `[0.01, 0.1, 1, 10, 100]`.
4. Build a learning curve (training size vs. CV accuracy) for the Random Forest on Iris using `sklearn.model_selection.learning_curve`.

In [ ]:
# your work here
